In [4]:
# ============================================================
# AQUAINSIGHT - TASK 2
# Water Quality Index (WQI) Comparison
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

from tkinter import Tk, filedialog

# ------------------------------------------------------------
# 1. FIND AND LOAD THE DATASET
# ------------------------------------------------------------

FILE = "Water Quality.csv"

def find_csv_file(filename):
    # Check the current working directory
    if os.path.isfile(filename):
        return os.path.abspath(filename)

    # Check common Windows folders
    search_folders = [
        os.getcwd(),
        os.path.expanduser("~/Desktop"),
        os.path.expanduser("~/Downloads"),
        os.path.expanduser("~/Documents"),
    ]

    # Search folders and their subfolders
    for folder in search_folders:
        if os.path.isdir(folder):
            for root, dirs, files in os.walk(folder):
                # Skip common large/unnecessary directories
                dirs[:] = [
                    d for d in dirs
                    if d not in {
                        ".git", "node_modules",
                        "__pycache__", ".venv", "venv"
                    }
                ]

                for file in files:
                    if file.lower() == filename.lower():
                        return os.path.join(root, file)

    return None


file_path = find_csv_file(FILE)

# If the file wasn't found, let the user select it
if file_path is None:
    print(f"Could not automatically find {FILE!r}.")
    print("Please select your Water Quality CSV file.")

    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    file_path = filedialog.askopenfilename(
        title="Select Water Quality CSV",
        filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
    )

    root.destroy()

if not file_path:
    raise FileNotFoundError(
        "No file selected. Please select your Water Quality CSV file."
    )

print("Dataset found:", file_path)

df = pd.read_csv(file_path, low_memory=False)

print("\nDataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nAvailable columns:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 2. CLEAN COLUMN NAMES AND CONVERT DATA TYPES
# ------------------------------------------------------------

df.columns = df.columns.str.strip()

required_columns = [
    "MonitoringLocationID",
    "ActivityStartDate",
    "CharacteristicName",
    "ResultValue",
    "ResultUnit"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}\n"
        "Please check that you selected the correct dataset."
    )

df["ActivityStartDate"] = pd.to_datetime(
    df["ActivityStartDate"], errors="coerce"
)

df["ResultValue"] = pd.to_numeric(
    df["ResultValue"], errors="coerce"
)

df["CharacteristicName"] = (
    df["CharacteristicName"].astype("string").str.strip()
)

df["MonitoringLocationID"] = (
    df["MonitoringLocationID"].astype("string").str.strip()
)

df["ResultUnit"] = df["ResultUnit"].astype("string").str.strip()

df = df.dropna(
    subset=[
        "MonitoringLocationID",
        "ActivityStartDate",
        "CharacteristicName",
        "ResultValue"
    ]
).copy()

# Monthly grouping period
df["YearMonth"] = df["ActivityStartDate"].dt.to_period("M").astype(str)

print("\nRows remaining after basic cleaning:", len(df))


# ------------------------------------------------------------
# 3. CREATE A MONTHLY WATER QUALITY FEATURE TABLE
# ------------------------------------------------------------

# This creates monthly median values for each location and parameter.
# Different parameters may come from different physical samples,
# so this table is for exploratory comparison.

group_columns = [
    "MonitoringLocationID",
    "YearMonth",
    "CharacteristicName"
]

monthly = (
    df.groupby(group_columns, dropna=False)["ResultValue"]
      .median()
      .reset_index()
)

monthly_wide = monthly.pivot_table(
    index=["MonitoringLocationID", "YearMonth"],
    columns="CharacteristicName",
    values="ResultValue",
    aggfunc="first"
).reset_index()

monthly_wide.columns.name = None

print("\nMonthly feature table created.")
print("Rows:", len(monthly_wide))
print("Parameters available:", len(monthly_wide.columns) - 2)


# ------------------------------------------------------------
# 4. SAVE THE MONTHLY FEATURE TABLE
# ------------------------------------------------------------

output_folder = os.path.join(
    os.path.dirname(file_path),
    "task2_wqi_outputs"
)

os.makedirs(output_folder, exist_ok=True)

monthly_output = os.path.join(
    output_folder,
    "monthly_water_quality_features.csv"
)

monthly_wide.to_csv(monthly_output, index=False)

print("\nMonthly feature table saved to:")
print(monthly_output)


# ------------------------------------------------------------
# 5. CREATE A WQI SPECIFICATION TEMPLATE
# ------------------------------------------------------------

# IMPORTANT:
# A scientifically defensible WQI requires a defined method,
# parameter standards, ideal values (where applicable), and weights.
# These values must come from your project's approved specification.
#
# This template is NOT a recommended set of standards or weights.

template_path = os.path.join(
    output_folder,
    "wqi_specification_template.csv"
)

if not os.path.exists(template_path):
    template = pd.DataFrame({
        "Parameter": ["pH"],
        "Weight": [1.0],
        "IdealValue": [7.0],
        "StandardValue": [8.5]
    })

    template.to_csv(template_path, index=False)

print("\nWQI specification template:")
print(template_path)


# ------------------------------------------------------------
# 6. CALCULATE WQI ONLY IF A VALID SPECIFICATION IS PROVIDED
# ------------------------------------------------------------

spec_path = os.path.join(
    os.path.dirname(file_path),
    "wqi_specification.csv"
)

def calculate_configurable_wqi(data, specification):
    """
    Generic weighted arithmetic WQI.

    This is not automatically an NSF-WQI or a regulatory compliance
    assessment. Use only with an approved specification.

    Formula:
        Quality rating Q_i = 100 * (V_i - I_i) / (S_i - I_i)
        WQI = sum(Q_i * W_i) / sum(W_i)

    For parameters where the approved method uses a different formula
    (for example, dissolved oxygen), implement that method separately.
    """

    required_spec_columns = [
        "Parameter",
        "Weight",
        "IdealValue",
        "StandardValue"
    ]

    missing = [
        col for col in required_spec_columns
        if col not in specification.columns
    ]

    if missing:
        raise ValueError(
            f"Specification is missing columns: {missing}"
        )

    specification = specification.copy()
    specification["Parameter"] = (
        specification["Parameter"].astype(str).str.strip()
    )

    for col in ["Weight", "IdealValue", "StandardValue"]:
        specification[col] = pd.to_numeric(
            specification[col], errors="coerce"
        )

    specification = specification.dropna(
        subset=required_spec_columns
    )

    if specification.empty:
        raise ValueError("The WQI specification contains no valid rows.")

    if (specification["Weight"] <= 0).any():
        raise ValueError("All parameter weights must be greater than zero.")

    if (
        specification["StandardValue"]
        == specification["IdealValue"]
    ).any():
        raise ValueError(
            "StandardValue and IdealValue must differ "
            "for this generic formula."
        )

    available_parameters = set(data.columns)

    valid_spec = specification[
        specification["Parameter"].isin(available_parameters)
    ].copy()

    if valid_spec.empty:
        raise ValueError(
            "None of the parameters in the specification match "
            "the dataset's CharacteristicName values."
        )

    result = data[["MonitoringLocationID", "YearMonth"]].copy()
    weighted_sum = pd.Series(0.0, index=data.index)
    weight_sum = pd.Series(0.0, index=data.index)

    for _, row in valid_spec.iterrows():
        parameter = row["Parameter"]
        weight = float(row["Weight"])
        ideal = float(row["IdealValue"])
        standard = float(row["StandardValue"])

        values = pd.to_numeric(data[parameter], errors="coerce")

        quality_rating = (
            100.0 * (values - ideal) / (standard - ideal)
        )

        valid = values.notna() & np.isfinite(quality_rating)

        weighted_sum.loc[valid] += (
            quality_rating.loc[valid] * weight
        )
        weight_sum.loc[valid] += weight

    result["WQI"] = weighted_sum.div(
        weight_sum.replace(0, np.nan)
    )

    result["ParametersUsed"] = weight_sum.map(
        lambda x: int(round(x)) if pd.notna(x) else 0
    )

    return result


if os.path.exists(spec_path):
    print("\nFound WQI specification:", spec_path)

    specification = pd.read_csv(spec_path)

    wqi_result = calculate_configurable_wqi(
        monthly_wide,
        specification
    )

    wqi_output = os.path.join(output_folder, "monthly_wqi_results.csv")
    wqi_result.to_csv(wqi_output, index=False)

    print("\nWQI results saved to:")
    print(wqi_output)
    print("\nPreview:")
    print(wqi_result.head(10))

else:
    print("\nWQI was NOT calculated.")
    print("No approved WQI specification was found.")
    print(
        "Review the template, then create 'wqi_specification.csv' "
        "using the method, weights, and standards approved for your project."
    )


# ------------------------------------------------------------
# 7. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("TASK 2 COMPLETED")
print("=" * 55)
print("Input dataset:", file_path)
print("Monthly feature table:", monthly_output)
print("WQI specification template:", template_path)

if os.path.exists(spec_path):
    print("WQI output:", wqi_output)
else:
    print("WQI output: Not generated (approved specification required).")

Dataset found: C:\Users\YASH/Desktop\BrainyBeam\task 2 web\Water Quality.csv

Dataset loaded successfully.
Rows: 229916
Columns: 40

Available columns:
['DatasetName', 'MonitoringLocationID', 'MonitoringLocationName', 'MonitoringLocationLatitude', 'MonitoringLocationLongitude', 'MonitoringLocationHorizontalCoordinateReferenceSystem', 'MonitoringLocationHorizontalAccuracyMeasure', 'MonitoringLocationHorizontalAccuracyUnit', 'MonitoringLocationVerticalMeasure', 'MonitoringLocationVerticalUnit', 'MonitoringLocationType', 'ActivityType', 'ActivityMediaName', 'ActivityStartDate', 'ActivityStartTime', 'ActivityEndDate', 'ActivityEndTime', 'ActivityDepthHeightMeasure', 'ActivityDepthHeightUnit', 'SampleCollectionEquipmentName', 'CharacteristicName', 'MethodSpeciation', 'ResultSampleFraction', 'ResultValue', 'ResultUnit', 'ResultValueType', 'ResultDetectionCondition', 'ResultDetectionQuantitationLimitMeasure', 'ResultDetectionQuantitationLimitUnit', 'ResultDetectionQuantitationLimitType', 'Res